In [1]:
import pandas as pd
import numpy  as np
from pytorch_tabnet.tab_model import TabNetRegressor
from sklearn.model_selection import KFold
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score, root_mean_squared_error
from sklearn.model_selection import train_test_split

In [2]:
DATA_PATH = r'../../data/entraineddropletfraction_VerticalFlow.csv'

In [3]:
import warnings, math, numpy as np, pandas as pd
warnings.filterwarnings("ignore")

df = pd.read_csv(DATA_PATH)
print("Shape:", df.shape)
display(df.head())

TARGET_COL = None
for c in df.columns:
    lc = c.lower()
    if any(k in lc for k in ["entrain","fraction"]) or lc in ["e","entrained","entrainment","target","y"]:
        if pd.api.types.is_numeric_dtype(df[c]):
            TARGET_COL = c
            break
if TARGET_COL is None:
    for c in reversed(df.columns):
        if pd.api.types.is_numeric_dtype(df[c]):
            TARGET_COL = c
            break

print("Detected target column:", TARGET_COL)

Shape: (1367, 9)


,D,Usg,UsL,rhol,rhog,st,MuL,Mug,e
0,0.1016,18.39,0.1,997.0,1.184,0.07,0.000894,0.000019,0.63
1,0.1016,23.66,0.1,997.0,1.184,0.07,0.000894,0.000019,0.61
2,0.1016,28.87,0.1,997.0,1.184,0.07,0.000894,0.000019,0.67
3,0.1016,9.17,0.2,997.0,1.184,0.07,0.000894,0.000019,0.64
4,0.1016,12.08,0.2,997.0,1.184,0.07,0.000894,0.000019,0.63


Detected target column: e


In [4]:
# Define features and target
features = ['D',	'Usg',	'UsL',	'rhol',	'rhog',	'st',	'MuL',	'Mug']
target = 'e'

In [5]:
X = df[features]
y = df[target]

In [6]:
# --- Split the data ---

#Split data into training and testing sets
X = pd.DataFrame(df, columns=df.columns.difference(['e']))
y = pd.Series(df['e'], name='e')
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

#X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

In [7]:
# --- Prepare Data for TabNet ---
X_train_np, y_train_np = X_train.values, y_train.values.reshape(-1, 1)
X_test_np, y_test_np = X_test.values, y_test.values.reshape(-1, 1)


In [8]:
import torch
tabnet = TabNetRegressor(
        n_d=16,             # decision layer width
        n_a=16,             # attention layer width
        n_steps=5,          # number of steps
        gamma=1.5,          # relaxation parameter
        n_independent=2,    # independent layers
        n_shared=2,         # shared layers
        optimizer_params=dict(lr=2e-2),
        scheduler_params={"step_size":50, "gamma":0.9},
        scheduler_fn=torch.optim.lr_scheduler.StepLR,
        mask_type='entmax', # sparsemax or entmax
        verbose=1,
        seed=42
    )

    # -----------------------------
    # 3. TRAIN TABNET
    # -----------------------------
tabnet.fit(
        X_train_np, y_train_np,
        eval_set=[(X_train_np, y_train_np), (X_test_np, y_test_np)],
        eval_name=['train','valid'],
        max_epochs=500,
        patience=50,
        batch_size=128,
        virtual_batch_size=64,
        num_workers=0
    )

epoch 0  | loss: 1.0707  | train_mse: 334.3989| valid_mse: 389.92758|  0:00:00s
epoch 1  | loss: 0.33751 | train_mse: 49.27908| valid_mse: 50.04978|  0:00:01s
epoch 2  | loss: 0.22411 | train_mse: 5.06833 | valid_mse: 3.64765 |  0:00:01s
epoch 3  | loss: 0.12685 | train_mse: 14.12254| valid_mse: 14.27298|  0:00:02s
epoch 4  | loss: 0.08277 | train_mse: 7.98662 | valid_mse: 8.20996 |  0:00:02s
epoch 5  | loss: 0.06681 | train_mse: 5.8568  | valid_mse: 5.93724 |  0:00:02s
epoch 6  | loss: 0.06357 | train_mse: 2.66665 | valid_mse: 2.69508 |  0:00:03s
epoch 7  | loss: 0.05485 | train_mse: 0.22744 | valid_mse: 0.22035 |  0:00:03s
epoch 8  | loss: 0.05093 | train_mse: 1.0994  | valid_mse: 1.13576 |  0:00:03s
epoch 9  | loss: 0.05047 | train_mse: 0.54647 | valid_mse: 0.56534 |  0:00:04s
epoch 10 | loss: 0.04574 | train_mse: 0.24575 | valid_mse: 0.24412 |  0:00:04s
epoch 11 | loss: 0.05122 | train_mse: 0.15226 | valid_mse: 0.14587 |  0:00:04s
epoch 12 | loss: 0.04934 | train_mse: 0.21871 | val

In [9]:
##Evaluate the performance of the predictions using the following metrics: Mean Absolute Error (MAE), Mean Squared Error (MSE), Root Mean Squared Error (RMSE), R-squared, and Mean Absolute Percentage Error (MAPE).
preds = tabnet.predict(X_test_np).flatten()

print("R²:", r2_score(y_test_np, preds))
print("MAE:", mean_absolute_error(y_test_np, preds))
print("RMSE:", root_mean_squared_error(y_test_np, preds))
print("MAPE:", np.mean(np.abs((y_test_np - preds) / y_test_np)) * 100)

R²: 0.8549703939943557
MAE: 0.07538637678728824
RMSE: 0.10813787053992106
MAPE: 231.54382365606997
